# 00 - Environment check

Run this first on a fresh pod. It fails loudly here if something is wrong, which is
much cheaper than finding out three hours into a training run.

Checks: the package imports, a GPU is visible, torch was built against CUDA, and all
three backbones can be constructed and take a forward pass.

In [1]:
import json, sys, time
from pathlib import Path

# Works whether the kernel starts in notebooks/ or at the repo root.
ROOT = Path.cwd()
while not (ROOT / "src" / "ham10000").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

DATA = ROOT / "data"
RUNS = ROOT / "runs"
print("repo:", ROOT)

repo: /workspace/ham10000-cnn-comparison


In [2]:
import torch, torchvision, sklearn, pandas, numpy, matplotlib, statsmodels

for name, mod in [("torch", torch), ("torchvision", torchvision), ("sklearn", sklearn),
                  ("pandas", pandas), ("numpy", numpy), ("matplotlib", matplotlib),
                  ("statsmodels", statsmodels)]:
    print(f"{name:<14} {mod.__version__}")

torch          2.8.0+cu128
torchvision    0.23.0+cu128
sklearn        1.9.0
pandas         3.0.5
numpy          2.1.2
matplotlib     3.11.1
statsmodels    0.14.6


## GPU

If `cuda available` is False, stop. Training on CPU is not viable for twelve runs, and
the whole point of the RunPod session is the GPU.

In [3]:
print("cuda available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device         :", torch.cuda.get_device_name(0))
    print("capability     :", torch.cuda.get_device_capability(0))
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"memory         : {total:.1f} GB")
else:
    raise SystemExit("No GPU visible. Check the pod before going any further.")

cuda available : True
device         : NVIDIA RTX A4500
capability     : (8, 6)
memory         : 21.0 GB


## Backbones

Constructs each of the three and pushes one batch through. This also downloads the
ImageNet weights, so doing it now means the training notebooks do not stall on a
download mid-run.

The dropout column is worth a look: the three backbones do not agree, and that
asymmetry is declared rather than papered over.

In [4]:
from ham10000.models import build_model, describe_model

for arch in ["mobilenet_v2", "resnet50", "efficientnet_b3"]:
    info = describe_model(arch)
    print(f"{arch:<18} {info['trainable_params']:>11,} params  "
          f"{info['weight_layers']:>3} weight layers  dropout={info['dropout_p']}")

mobilenet_v2         2,232,839 params   53 weight layers  dropout=[0.2]
resnet50            23,522,375 params   54 weight layers  dropout=[]
efficientnet_b3     10,706,991 params  131 weight layers  dropout=[0.3]


In [5]:
device = torch.device("cuda")
for arch, size in [("mobilenet_v2", 224), ("resnet50", 224),
                   ("efficientnet_b3", 224), ("efficientnet_b3", 300)]:
    model = build_model(arch).to(device).eval()
    with torch.no_grad():
        out = model(torch.randn(2, 3, size, size, device=device))
    print(f"{arch:<18} @{size}  ->  {tuple(out.shape)}")
    del model
torch.cuda.empty_cache()
print("\nAll three backbones build and run.")

mobilenet_v2       @224  ->  (2, 7)
resnet50           @224  ->  (2, 7)
efficientnet_b3    @224  ->  (2, 7)
efficientnet_b3    @300  ->  (2, 7)

All three backbones build and run.


## Where the data should be

The next notebook handles getting it there. This just reports the current state.

In [6]:
meta = DATA / "HAM10000_metadata.csv"
print("metadata present :", meta.exists())
for folder in ["HAM10000_images_part_1", "HAM10000_images_part_2"]:
    p = DATA / folder
    n = len(list(p.glob("*.jpg"))) if p.exists() else 0
    print(f"{folder:<26} {n:>6} images")

metadata present : False
HAM10000_images_part_1          0 images
HAM10000_images_part_2          0 images
